In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import random
from collections import Counter

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(42)

# Dataset path
dataset_path = "/kaggle/input/datasets/ninadaithal/imagesoasis/Data"
print("Setup complete ✓")
print(f"GPU available: {torch.cuda.is_available()}")

Setup complete ✓
GPU available: True


In [4]:
# Image size for ViT
IMG_SIZE = 224

# Transforms for training (with augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transforms for validation/test (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Transforms defined ✓")
print(f"\nTraining transforms: {len(train_transforms.transforms)} steps")
print(f"Validation transforms: {len(val_transforms.transforms)} steps")

Transforms defined ✓

Training transforms: 6 steps
Validation transforms: 3 steps


In [5]:
class AlzheimerDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Load all image paths and labels
        for cls in self.classes:
            cls_path = os.path.join(data_dir, cls)
            for img_name in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_name))
                self.labels.append(self.class_to_idx[cls])
        
        print(f"Dataset loaded: {len(self.images)} images")
        print(f"Classes: {self.class_to_idx}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Test it
full_dataset = AlzheimerDataset(dataset_path, transform=val_transforms)

Dataset loaded: 86437 images
Classes: {'Mild Dementia': 0, 'Moderate Dementia': 1, 'Non Demented': 2, 'Very mild Dementia': 3}


In [6]:
from torch.utils.data import random_split

# Split ratios: 70% train, 15% val, 15% test
total = len(full_dataset)
train_size = int(0.70 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Total images:      {total:,}")
print(f"Training set:      {len(train_dataset):,} ({len(train_dataset)/total*100:.1f}%)")
print(f"Validation set:    {len(val_dataset):,} ({len(val_dataset)/total*100:.1f}%)")
print(f"Test set:          {len(test_dataset):,} ({len(test_dataset)/total*100:.1f}%)")

Total images:      86,437
Training set:      60,505 (70.0%)
Validation set:    12,965 (15.0%)
Test set:          12,967 (15.0%)


In [7]:
# Get labels for training set only
train_labels = [full_dataset.labels[i] for i in train_dataset.indices]

# Count per class in training set
class_counts = Counter(train_labels)
print("Training set class distribution:")
for cls_name, idx in full_dataset.class_to_idx.items():
    print(f"  {cls_name}: {class_counts[idx]:,}")

# Compute weight per class (inverse frequency)
total_train = len(train_labels)
class_weights = {cls: total_train / count 
                 for cls, count in class_counts.items()}

# Assign weight to each sample
sample_weights = [class_weights[label] for label in train_labels]
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

# WeightedRandomSampler — minority classes get sampled more often
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nClass weights (higher = sampled more often):")
for cls_name, idx in full_dataset.class_to_idx.items():
    print(f"  {cls_name}: {class_weights[idx]:.2f}")

Training set class distribution:
  Mild Dementia: 3,440
  Moderate Dementia: 313
  Non Demented: 47,103
  Very mild Dementia: 9,649

Class weights (higher = sampled more often):
  Mild Dementia: 17.59
  Moderate Dementia: 193.31
  Non Demented: 1.28
  Very mild Dementia: 6.27


In [8]:
# Apply correct transforms to each split
train_dataset.dataset.transform = train_transforms

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,  # weighted sampler handles shuffling
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")
print(f"Test batches:  {len(test_loader):,}")

# Verify one batch
images, labels = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  Images shape: {images.shape}")
print(f"  Labels shape: {labels.shape}")
print(f"  Pixel range: [{images.min():.2f}, {images.max():.2f}]")

Batch size: 32
Train batches: 1,891
Val batches:   406
Test batches:  406

Sample batch:
  Images shape: torch.Size([32, 3, 224, 224])
  Labels shape: torch.Size([32])
  Pixel range: [-2.12, 2.64]


In [9]:
# Check if sampler is actually balancing classes
sample_labels = []
for _, labels in train_loader:
    sample_labels.extend(labels.tolist())
    if len(sample_labels) >= 5000:
        break

# Count sampled classes
sampled_counts = Counter(sample_labels[:5000])
idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

print("Class distribution after weighted sampling (first 5000 samples):")
for idx in sorted(sampled_counts.keys()):
    count = sampled_counts[idx]
    print(f"  {idx_to_class[idx]:<25} {count:>6} ({count/50:.1f}%)")

Class distribution after weighted sampling (first 5000 samples):
  Mild Dementia               1257 (25.1%)
  Moderate Dementia           1236 (24.7%)
  Non Demented                1239 (24.8%)
  Very mild Dementia          1268 (25.4%)


# 02 — Preprocessing Pipeline

## Steps Implemented

1. **Transforms** — Resize to 224×224, augmentation for training 
   (flip, rotation, color jitter), normalization using ImageNet stats
2. **Custom Dataset** — AlzheimerDataset class loading 86,437 images
3. **Train/Val/Test Split** — 70% / 15% / 15% with fixed seed (42)
4. **Class Imbalance Fix** — WeightedRandomSampler with inverse 
   frequency weights (Moderate Dementia weight: 193.31x)
5. **DataLoaders** — Batch size 32, verified shape [32, 3, 224, 224]

## Result
Class distribution after sampling: ~25% per class (from original 137.8x imbalance)
Model is now ready to learn all four dementia stages equally.

## Next
03_baseline_cnn.ipynb — establish performance benchmark before ViT